### 📦 Offline Dependency Installation

- Installs `datasets` and `trl` into a local directory (`/kaggle/working/packages`)
- Prefers **offline packages** (for reproducibility & no internet dependency)
- Falls back to PyPI if offline packages are unavailable
- Appends install path to `sys.path`

✅ Ensures Kaggle-compatible, reproducible environment  
⚠️ Avoids permission issues with global installs

In [ ]:
import subprocess, sys, os
from pathlib import Path
def resolve_python_path(target_dir):
    for pth_file in Path(target_dir).glob("*.pth"):
        with pth_file.open() as fp:
            relpath = fp.read()
            rel_pack_path = (pth_file.parent/relpath)
            if rel_pack_path.exists():
                print(f"append {rel_pack_path}")
                sys.path.append(str(rel_pack_path))



offline_dir = "/kaggle/input/nvidia-nemotron-offline-packages/offline_packages"
target_dir = "/kaggle/working/packages"

os.makedirs(target_dir, exist_ok=True)
resolve_python_path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/")

if os.path.exists(offline_dir):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index",
        "--find-links", offline_dir,
        "--target", target_dir,
        "datasets", "trl"
    ])
    print("Installed from offline packages")


# Add to Python path
sys.path.append(target_dir)
resolve_python_path(target_dir)

import datasets, cutlass

### ⚙️ Environment Setup & Libraries

- Enables `expandable_segments` → reduces CUDA memory fragmentation (important for 30B)
- Imports:
  - `torch`, `transformers` → model training
  - `polars` → fast data loading
  - `datasets` → HF dataset wrapper
  - `trl`, `peft` → SFT + LoRA fine-tuning
  - `kagglehub` → model download

✅ Optimized for large-model training on H100

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import sys, stat, shutil, gc, zipfile
import polars as pl
import torch
import torch.nn.functional as F
import kagglehub
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

### 🛠️ Triton + RMSNorm Fix (Kaggle Hack)

- Replaces `rmsnorm_fn` with a pure PyTorch implementation
  → avoids Triton kernel crashes

- Copies & patches `ptxas` binary for Blackwell compatibility
- Overrides Triton backend paths + permissions

✅ Fixes incompatibilities between:
  - Kaggle runtime
  - Triton compiler
  - Nemotron kernels

⚠️ Hacky but necessary for stable training

In [ ]:
# === Triton fixes ===
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast: x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None: out = out + bias.float()
    if z is not None: out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

# WARNING: This patch is strictly for Kaggle's environment.
src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"] = dst
    print("Triton ptxas fix applied.")

### 📊 Training Config + Prebuilt Rule-Based Dataset

**Hyperparameters:**
- LoRA rank = 32
- Seq length = 2048 → supports reasoning chains
- Batch size = 1 + grad accumulation = 4
- LR = 1e-4, epochs = 2

**Data:**
- Loads a prebuilt external CSV generated from solver-verified rows
- Expects 600 rows already curated in baseline-compatible schema
- Converts the CSV into a HuggingFace Dataset

✅ Keeps the original baseline training pipeline
✅ Changes only the training data source

In [ ]:
# === Hyperparameters ===
SUBSAMPLE_SIZE = 600
LORA_RANK = 32
MAX_SEQ_LEN = 2048     # Increased from 1024 to support reasoning chains
NUM_EPOCHS = 2         # Increased for better convergence
BATCH_SIZE = 1
GRAD_ACCUM = 4         
LR = 1e-4
OUTPUT_DIR = "/kaggle/working/adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# === Data ===
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
DATASET_CSV = "/kaggle/input/datasets/renta0426/rule-based-training-data/rule_based_verified_600_training_data.csv"
CSV_SCHEMA = {
    "id": pl.Utf8,
    "prompt": pl.Utf8,
    "answer": pl.Utf8,
    "generated_cot": pl.Utf8,
    "label": pl.Utf8,
}
train_df = pl.read_csv(DATASET_CSV, schema_overrides=CSV_SCHEMA)
if len(train_df) != SUBSAMPLE_SIZE:
    raise ValueError(f"Expected {SUBSAMPLE_SIZE} rows in {DATASET_CSV}, found {len(train_df)}")
hf_dataset = Dataset.from_pandas(train_df.to_pandas())

### 💬 Training Prompt Construction

- Converts dataset into chat format:
  - User: problem + instruction (`\\boxed{}`)
  - Assistant: short verified reasoning trace + final boxed answer

- Uses `tokenizer.apply_chat_template`
  → ensures model-native formatting

- Fallback to manual template if needed

⚠️ Critical assumptions:
- The CSV already follows `id,prompt,answer,generated_cot,label`
- `generated_cot` contains the prebuilt rule-based trace
- `answer` contains the final answer only

✅ Aligns training with the same inference-style answer format as baseline

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_training_text(example):
    prompt = example["prompt"]
    answer = example["answer"] 
    cot = example["generated_cot"] # Extract the prebuilt rule-based reasoning trace
    
    user_msg = prompt + "\nPut your final answer inside \\boxed{}."
    
    # Combine the trace with the final answer
    assistant_msg = f"{cot}\n\n\\boxed{{{answer}}}" 
    
    try:
        messages = [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_msg},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    except Exception:
        text = (
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n{assistant_msg}<|im_end|>"
        )
    return {"text": text}

# Apply mapping
hf_dataset = hf_dataset.map(build_training_text, remove_columns=hf_dataset.column_names)

### 🧠 Model Loading & LoRA Fine-tuning

**Model:**
- Loads Nemotron-3 Nano 30B (bf16, GPU)
- Enables gradient checkpointing → saves VRAM

**Patching:**
- Disables fast path kernels (stability fix)

**LoRA:**
- Rank = 32, alpha = 32
- Targets all major projection layers
- Reduces trainable params significantly

**Training:**
- Uses `SFTTrainer` (TRL)
- Cosine LR scheduler + warmup
- No checkpoint saving (Kaggle constraint)

✅ Efficient fine-tuning of 30B on single H100

In [ ]:
# === Model ===
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, 
    device_map={"": 0}, 
    trust_remote_code=True, 
    dtype=torch.bfloat16,
)
model.gradient_checkpointing_enable()

for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}: is_fast_path_available = False")

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# === Triton compiler fix ===
import triton.backends.nvidia.compiler as nv_compiler
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
nv_compiler.get_ptxas_version = lambda arch: "12.0"

# === Training ===
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE, # Optimized
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    logging_steps=5,
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    save_strategy="no",
    report_to="none",
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    processing_class=tokenizer,
    args=training_args,
)

print("Starting training...")
trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR) # Ensure tokenizer config is saved

print(f"Adapter saved to {OUTPUT_DIR}:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f} ({size/1024:.1f} KB)")

### 💾 Save LoRA Adapter

- Saves:
  - `adapter_model.safetensors`
  - `adapter_config.json`
  - tokenizer config

- Prints file sizes for verification

✅ Only LoRA weights saved (NOT full model)
→ lightweight (~MBs instead of GBs)

### 📦 Create Submission ZIP

- Compresses adapter files into `submission.zip`
- Verifies required files exist:
  - `adapter_config.json`
  - `adapter_model.safetensors`

- Raises error if missing (prevents submission failure)

✅ Ready for Kaggle submission
⚠️ Missing files = instant evaluation failure

In [ ]:
import zipfile
import os

# Define paths
OUTPUT_DIR = "/kaggle/working/adapter"
zip_path = "/kaggle/working/submission.zip"

print(f"Packaging files from {OUTPUT_DIR}...")

# Create the zip file with compression
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        zf.write(fpath, fname) 

print(f"Created {zip_path} ({os.path.getsize(zip_path) / 1024 / 1024:.1f} MB)")

with zipfile.ZipFile(zip_path, 'r') as zf:
    zip_contents = zf.namelist()
    print(f"Zip Contents: {zip_contents}")
    
    if "adapter_config.json" not in zip_contents:
        raise AssertionError("CRITICAL ERROR: adapter_config.json is missing from the zip. The Kaggle evaluation will fail.")
    if "adapter_model.safetensors" not in zip_contents:
         raise AssertionError("CRITICAL ERROR: adapter_model.safetensors is missing from the zip.")

print("✅ submission.zip is ready! You can now submit this file to the competition.")